These notebooks document the process of estimating subjective landscape beauty on a raster level for all of Europe. 

To acchieve this we replicate the work of [cite German paper]. 

Libraries, workflow and data sources

Several python and R libraries are used to run this document. 
Spatial data: terra

Data Visualization: leaflet 

Raster size, projection and alignment are chosen to fit the German reference data. 
1x1 km
EPSG: 3035

# About the German beauty model

In 2021 Roth et al. published a dataset, assigning a value of landscape beauty to each cell of a 1 by 1 kilometer grid in Germany. 
The basis for this map were user-rated landscape photos of Germany.  These photos were taken as part of a photo documentation project in 30 representative sampling areas, each approximately 150 km² in size, across Germany. The sampling areas represent the diversity of German landscapes, from coastal regions to the Alps, and from forested areas to rural regions and urban centers. An expert-selected set of over 800 landscape photos was evaluated through an online survey in collaboration with a citizen science panel of over 3,500 participants.

These images are analzed using GIS tools to extract the visable landscape features. These features include landcover, landuse and relief energy.
The features are chosen, such that a connection to a countrywide dataset there can be drawn.
This feature extraction also notes the distance of said features from the camera. For this purpose, the authors define 4 zones:

zone 1) 0 to 500m

zone 2) 500m to 2.000m

zone 3) 2.000m to 4.000m

zone 4) 4.000m to 10.000m

The link between user ratings and landscape features are then modeled using ordinary least square regressions.

The final map are then the prediciton of these models using the countrywide available datasets. 

Only the final output maps are open to the public unfortunatly. In order to acchieve our goal of extending these models to the rest of Europe, we first need to replicate the model as best as we can, based on the documentation, and then apply it to datasets that are available for the whole of the EU. 

In [ ]:
# leaflet plot the beauty map. 

# Exploring the model

The authors apply well established feature selection methods to reduce the initial 80+ covariates down to 17 for beauty. 

(* = significant (α = 0.05); ** = highly significant (α = 0.001)).

| No. | Regressor                                              | Area of Influence                 | Unstandardized Coefficients         | Standardized Coefficients (Beta)    |
|-----|--------------------------------------------------------|------------------------------------|--------------------------------------|-------------------------------------|
| 1   | Constant                                               | /                                  | 7.109                                | /                                   |
| 2   | Terrain Height Difference 1                            | 0 to 2,000 m (Zones 1 and 2)       | + 0.002**                            | + 0.171**                           |
| 3   | Terrain Height Difference 2                            | 2,000 to 10,000 m (Zones 3 and 4)  | + 0.001**                            | + 0.185**                           |
| 4   | Lakes, Seas, Watercourses                              | 0 to 500 m (Zone 1)                | + 0.008**                            | + 0.152**                           |
| 5   | Orchards                                               | Entire area of influence           | + 0.031**                            | + 0.096**                           |
| 6   | Forest                                                 | Entire area of influence           | + 0.005*                             | + 0.088*                            |
| 7   | Natural Grassland                                      | 500 to 2,000 m (Zone 2)            | + 0.025**                            | + 0.083**                           |
| 8   | Heaths, Moorlands                                      | 0 to 500 m (Zone 1)                | + 0.017*                             | + 0.068*                            |
| 9   | Degree of Hemoroby                                     | 0 to 500 m (Zone 1)                | - 0.317**                            | - 0.200**                           |
| 10  | Road Density                                           | 0 to 2,000 m (Zones 1 and 2)       | - 0.0001**                           | - 0.189**                           |
| 11  | Arable Land                                            | Entire area of influence           | - 0.010**                            | - 0.187**                           |
| 12  | Industrial, Commercial, Traffic, Mining, Landfills, Construction Areas 1 | 0 to 500 m (Zone 1)               | - 0.019**                            | - 0.187**                           |
| 13  | Industrial, Commercial, Traffic, Mining, Landfills, Construction Areas 2 | 500 to 2,000 m (Zone 2)           | - 0.018**                            | - 0.106**                           |
| 14  | Industrial, Commercial, Traffic, Mining, Landfills, Construction Areas 3 | 2,000 to 5,000 m (Zone 3)         | - 0.011*                             | - 0.065*                            |
| 15  | Power Line Density                                     | 0 to 500 m (Zone 1)                | - 0.0001**                           | - 0.101**                           |
| 16  | Sports and Recreation Facilities                       | 0 to 500 m (Zone 1)                | - 0.019**                            | - 0.077**                           |
| 17  | Sparse Vegetation                                      | 500 to 2,000 m (Zone 2)            | - 0.205*                             | - 0.075*                            |
| 18  | Wind Turbine Density                                   | Entire area of influence           | - 0.588*                             | - 0.071*                            |


This table will form the basis for our model. Since the only data at our disposal are the final output maps, we only need to consider these covariates for our model.
Where possible we use the same datasets as the authors and apply the same transformations as them.

# Datasets

## Elevation

The authors use the "Digitales Geländemodell (DGM)" which is based on Airborn Laserscanning. 
We use the EU wide version digital elevation model over Europe (EU-DEM) published by eurostat.  [cite]

## Landcover and Landuse

The basis for the landcover related variables are the Corine Landcover Classes. 
Specifically in the model are:

| Name                                   | Corine Landcover Codes              |
|----------------------------------------|-------------------------------------|
| Orchards                               | 222                                 |
| Forest                                 | 311, 312, 313                      |
| Natural Grassland                      | 321                                 |
| Moors and Heathland                    | 322                                 |
| Arable Land                            | 211                                 |
| Industrial, Commercial, Traffic, Mining, Landfills, Construction | 121, 122, 123, 124, 131, 132, 133 |
| Sports and Recreation Facilities       | 142                                 |
| Sparse Vegetation                      | 333                                 |
| Lakes, Seas, Watercourses              | 512, 521, 522, 523                 |

     

We use the 2018 Corine Landcover classes at 100x100m resolution. 
Following the reference paper, we calculate the share of selected classes within each cell in our target raster (1x1km)

In [ ]:
# example image of Forest layer. 

# Open Street Maps

For road, wind turbine and powerpole density we use Open street maps, as do the authors of the reference paper. 

We downloaded an EU wide dataset published by geofabrik [cite] and extract the relevant features as vectors, which are then converted to density per cell. 
The points data (windturbines and powerpoles) are encoded as "number of elements in the cell", whereas the line data for roads is encoded as "total length within the cell"

## Neighborhood values